In [0]:
%sql
select * from bronze.shows limit 5

In [0]:
# tests/conftest.py
import pytest
from pyspark.sql import SparkSession

@pytest.fixture(scope="session")
def spark():
    spark = (
        SparkSession.builder
        .appName("pytest-pyspark")
        .master("local[2]")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )
    yield spark
    spark.stop()

Unit Tests Data Quality Rules  
Rule 1: Required Fields Must Not Be NULL

In [0]:
from pyspark.sql import SparkSession

spark_session = SparkSession.getActiveSession()
df_shows    = spark_session.table("silver.shows")
df_episodes = spark_session.table("silver.episodes")
df_cast     = spark_session.table("silver.cast")

In [0]:
# tests
def test_required_fields_not_null(shows_df):
    required_columns = ["show_id", "show_name", "runtime_minutes"]

    for col in required_columns:
        null_count = shows_df.filter(f"{col} IS NULL").count()
        assert null_count == 0, f"Column {col} contains NULL values"

In [0]:
# tests
from pyspark.sql.utils import AnalysisException

def test_required_fields_not_null(shows_df):
    required_columns = ["id", "name", "runtime"]

    for column in required_columns:
        try:
            # Check for NULL values
            null_count = shows_df.filter(f"{column} IS NULL").count()

            if null_count != 0:
                raise AssertionError(
                    f"Data quality check failed: Column '{column}' "
                    f"contains {null_count} NULL value(s)"
                )

        except AnalysisException as e:
            # Raised when column does not exist or Spark cannot analyze query
            raise AssertionError(
                f"Data quality check failed: Column '{column}' does not exist "
                f"or is not accessible in the DataFrame"
            ) from e

        except Exception as e:
            # Catch-all for unexpected Spark / runtime issues
            raise RuntimeError(
                f"Unexpected error while validating column '{column}': {str(e)}"
            ) from e

In [0]:
test_required_fields_not_null(df_shows.fillna({'runtime': 0}))

Rule 2: Runtime Must Be > 0

In [0]:
def test_runtime_greater_than_zero(shows_df):
    invalid_runtime = shows_df.filter("runtime <= 0").count()
    assert invalid_runtime == 0, "Runtime must be greater than 0"

In [0]:
from pyspark.sql.utils import AnalysisException

def test_runtime_greater_than_zero(shows_df):
    try:
        invalid_runtime = shows_df.filter("runtime <= 0").count()

        if invalid_runtime != 0:
            raise AssertionError(
                f"Data quality check failed: 'runtime' contains "
                f"{invalid_runtime} invalid value(s) (<= 0)"
            )

    except AnalysisException as e:
        # Column does not exist or Spark cannot analyze the query
        raise AssertionError(
            "Data quality check failed: Column 'runtime' does not exist "
            "or is not accessible in the DataFrame"
        ) from e

    except Exception as e:
        # Catch-all for unexpected Spark/runtime issues
        raise RuntimeError(
            f"Unexpected error while validating 'runtime' values: {str(e)}"
        ) from e


In [0]:
test_runtime_greater_than_zero(df_shows)

Rule 3: Show Names Must Be Unique Per ID

In [0]:
from pyspark.sql.functions import countDistinct

def test_show_name_unique_per_id(shows_df):
    duplicates = (
        shows_df
        .groupBy("id")
        .agg(countDistinct("name").alias("name_count"))
        .filter("name_count > 1")
        .count()
    )

    assert duplicates == 0, "Show ID has multiple show names"

In [0]:
from pyspark.sql.functions import countDistinct
from pyspark.sql.utils import AnalysisException

def test_show_name_unique_per_id(shows_df):
    try:
        duplicates = (
            shows_df
            .groupBy("id")
            .agg(countDistinct("name").alias("name_count"))
            .filter("name_count > 1")
            .count()
        )

        if duplicates != 0:
            raise AssertionError(
                f"Data quality check failed: {duplicates} show_id(s) "
                f"have multiple associated show names"
            )

    except AnalysisException as e:
        # Handles missing columns or Spark analysis errors
        raise AssertionError(
            "Data quality check failed: Required column(s) "
            "'id' or 'name' do not exist or are not accessible"
        ) from e

    except Exception as e:
        # Catch-all for unexpected Spark or runtime issues
        raise RuntimeError(
            f"Unexpected error while validating show name uniqueness per ID: {str(e)}"
        ) from e


In [0]:
test_show_name_unique_per_id(df_shows)

4. Delta Lake Expectations (Data Quality Enforcement)

Enforcing NOT NULL and Runtime > 0

In [0]:

from pyspark.sql.functions import col, sum as spark_sum

null_counts_df = df_shows.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_shows.columns
])
display(null_counts_df)

In [0]:
(
    df_shows.write
    .format("delta")
    .mode("overwrite")
    .option(
        "delta.constraints.valid_runtime",
        "runtime > 0"
    )
    .option(
        "delta.constraints.not_null_id",
        "id IS NOT NULL"
    )
    .saveAsTable("bronze.test_shows_delta_log")
)

In [0]:
(
    df_shows.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .option("enforceSchema", "true")
    .saveAsTable("bronze.test_shows_delta_log")
)

5. Schema Enforcement Test

In [0]:
# tests/test_schema_enforcement.py
import pytest
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


def test_schema_enforced(spark):
    wrong_schema = StructType([
        StructField("id", StringType(), True),
        StructField("name", IntegerType(), True),  # Wrong type
        StructField("runtime", IntegerType(), True)
    ])

    bad_df = spark.createDataFrame(
        [("1", 123, 45)],
        schema=wrong_schema
    )

    with pytest.raises(Exception):
        (
            bad_df.write
            .format("delta")
            .mode("append")
            .option("enforceSchema", "true")
            .saveAsTable("bronze.test_shows_delta_log")
        )

In [0]:
test_schema_enforced(df_shows)

6. Failure Simulation (Screenshot Deliverable)

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

bad_data = [
    ("1", "Breaking Bad", -10),  # Invalid runtime
    ("2", None, 55)              # Null show_name
]

bad_df = spark.createDataFrame(
    bad_data,
    ["id", "name", "runtime"]
)

In [0]:
from pyspark.sql.functions import col

df_shows = df_shows.withColumn("id", col("id").cast("string"))

In [0]:
bad_df.write \
    .format("delta") \
    .option("delta.constraints.valid_runtime", "runtime > 0") \
    .mode("append") \
    .saveAsTable("bronze.test_shows_delta_log")

In [0]:

(
    df_shows.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.test_shows_delta_log")
)
